# Product Feedback Analysis Using NLP
## Exploratory Data Analysis, Preprocessing, Feature Extraction & Sentiment Modeling

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure src directory is accessible
sys.path.append('..')
from src.preprocessing import preprocess_pipeline
from src.feature_extraction import build_tfidf_vectorizer
from src.evaluation import compute_metrics

### 1. Load Raw Dataset & Inspect Structure

In [ ]:
raw_csv = '../data/product_reviews.csv'
if not os.path.exists(raw_csv):
    raw_csv = '../amazon_reviews.csv'

df = pd.read_csv(raw_csv)
print("Dataset Shape:", df.shape)
df.head()

### 2. Rating Transformation (1-2 -> Negative, 3 -> Neutral, 4-5 -> Positive)

In [ ]:
def convert_rating(overall):
    if overall <= 2.0:
        return 'Negative'
    elif overall == 3.0:
        return 'Neutral'
    else:
        return 'Positive'

df['sentiment'] = df['overall'].apply(convert_rating)
df['sentiment'].value_counts()

### 3. Apply NLP Preprocessing Pipeline

In [ ]:
df['processed_text'] = df['reviewText'].astype(str).apply(preprocess_pipeline)
df[['reviewText', 'processed_text', 'sentiment']].head()

### 4. TF-IDF Feature Extraction & Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

X_train, X_test, y_train, y_test = train_test_split(
    df['processed_text'], df['sentiment'], test_size=0.2, random_state=42, stratify=df['sentiment']
)

vectorizer = build_tfidf_vectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Vocabulary size:", len(vectorizer.vocabulary_))

### 5. Model Training & Evaluation (Logistic Regression vs Naive Bayes)

In [ ]:
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_vec, y_train)
y_pred_lr = lr.predict(X_test_vec)

nb = MultinomialNB()
nb.fit(X_train_vec, y_train)
y_pred_nb = nb.predict(X_test_vec)

print("Logistic Regression Metrics:", compute_metrics(y_test, y_pred_lr))
print("Naive Bayes Metrics:", compute_metrics(y_test, y_pred_nb))